In [57]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker, read_sparse_h5, recompute_aggregate_scores

In [58]:
# data_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/"
# # data_name = ["E11_s1", "E13_s1", "E15_s1", "E18_s1"]
# data_name = ["Human_lymph_a1", "Human_lymph_d1"]
# adatas_rna = []
# adatas_adt = []
# for name in data_name:
#     if data_name == "Human_lymph_a1":
#         adata_rna = sc.read_h5ad(data_dir+f"/{name}/rna_filtered.h5ad")
#         adata_adt = sc.read_h5ad(data_dir+f"/{name}/adt_filtered.h5ad")
#     else:
#         adata_rna = sc.read_h5ad(data_dir+f"/{name}/rna.h5ad")
#         adata_adt = sc.read_h5ad(data_dir+f"/{name}/adt.h5ad")
#     if name == "Human_lymph_a1":
#         prefix = "A1"
#     else:
#         prefix = "D1"
#     adata_rna.obs_names = [prefix+"#"+i for i in adata_rna.obs_names]
#     adata_adt.obs_names = [prefix+"#"+i for i in adata_adt.obs_names]
#     adata_rna.var_names_make_unique()
#     adata_adt.var_names_make_unique()
#     adata_rna.obs["sample"] = name
#     adata_adt.obs["sample"] = name
#     adatas_rna.append(adata_rna)
#     adatas_adt.append(adata_adt)

In [59]:
# rna = sc.concat(adatas_rna)
# adt = sc.concat(adatas_adt)

In [60]:
# (rna.obs_names == adt.obs_names).all()

In [61]:
# rna.write("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_lymph/rna.h5ad")
# adt.write("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_lymph/adt.h5ad")

In [62]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_lymph/")

In [63]:
# rna = sc.read_h5ad("rna.h5ad")
# adt = sc.read_h5ad("adt.h5ad")

In [64]:
# sc.pp.filter_cells(rna, min_counts=1)
# sc.pp.filter_cells(adt, min_counts=1)

In [65]:
# rna = rna[adt.obs_names]

In [66]:
# rna.write("rna_filtered.h5ad")
# adt.write("adt_filtered.h5ad")

In [67]:
rna = sc.read_h5ad("rna_filtered.h5ad")
adt = sc.read_h5ad("adt_filtered.h5ad")

In [68]:
(rna.obs_names == adt.obs_names).all()

np.True_

In [69]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str, batch_key: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)
    batches = np.asarray(adata.obs[batch_key], dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))
        g.create_dataset("batches", data=np.array(batches, dtype="S"))


        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [70]:
h5ad_to_h5(rna, output_file="rna.h5", batch_key="sample")
h5ad_to_h5(adt, output_file="adt.h5", batch_key="sample")

In [71]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [72]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_lymph/"
# bm.run(methods=["MOFA2", "CellCharter"],#["MultiVI", "PRESENT", "MOFA2",  "scMDC", "Multigrate", "CellCharter", "TotalVI"],
#        RNA_file_path=data_folder+"rna.h5",
#        ADT_file_path=data_folder+"adt.h5",
#        n_cluster=11,
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_lymph/",
#        hvg_num=3000,
#        batch_key="batches"
#        )

In [73]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_lymph/cellcharter.csv", index_col=0)

In [74]:
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# rna.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# rna.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [75]:
# sc.pl.umap(rna, color=["sample", "cluster"])
# rna1 = rna[rna.obs["sample"]=="Human_lymph_a1"]
# rna2 = rna[rna.obs["sample"]=="Human_lymph_d1"]
# sc.pl.spatial(rna1, spot_size=1, color=["cluster"])
# sc.pl.spatial(rna2, spot_size=1, color=["cluster"])

Plot

In [76]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_lymph/"
# methods = bm.multi_omics_methods.copy()
methods = ["scMDC", "Multigrate", "MultiVI", "MOFA2", "CellCharter", "PRESENT", "TotalVI"]
res = bm.read_result(path=result_folder,
                     methods=methods,
                     reindex=False)

In [77]:
adt.obs["cell_type_merged"] = adt.obs["cell_type"].map({
    "Adipose Tissue": "Adipose Tissue",
    "Pericapsular adipose tissue": "Adipose Tissue",
    "Follicle": "Follicle",
    "B cell Follicle": "Follicle",
    "Subcapsular sinus": "Subcapsular sinus",
    "Marginal Sinus": "Subcapsular sinus",
    "Medulla cords": "Medulla cords",
    "Medullary Chords": "Medulla cords",
    "Medulla sinuses": "Medulla sinuses",
    "Medullary Sinus": "Medulla sinuses",
    "Capsule": "Capsule",
    "Cortex": "Cortex",
    "Paracortex": "Cortex",
    "Endothelial": "Medulla vessels",
    "Medulla vessels": "Medulla vessels",
    "Exclude": "Unknown",
    "Trabeculae": "Trabeculae",
    "Connective Tissue": "Connective Tissue",
    "Hilum": "Hilum",
})
rna = rna[adt.obs_names]
rna.obs["cell_type_merged"] = adt.obs["cell_type_merged"].copy()

In [78]:
len(set(adt.obs["cell_type_merged"]))

12

In [23]:
adt1 = adt[adt.obs["sample"]=="Human_lymph_a1"]
adt2 = adt[adt.obs["sample"]=="Human_lymph_d1"]

In [24]:
# sc.pl.embedding(adt1, basis="spatial", size=70, color="cell_type_merged")

In [25]:
# sc.pl.embedding(adt2, basis="spatial", size=70, color="cell_type_merged")

In [26]:
# for m in methods:
#     res = pd.read_csv(f"/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_lymph/{m.lower()}.csv", index_col=0)
#     res.columns = ["UMAP1", "UMAP2", "cluster"]
#     cluster = len(set(res["cluster"]))
#     print(m, cluster)

In [27]:
# res_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_lymph/"
# def search_resolution(adata, fixed_clus_count, increment=0.01):
#     closest_count = np.inf  
#     closest_res = None  
    
#     for res in sorted(list(np.arange(0.1, 1.5, increment)), reverse=True):
#         sc.tl.leiden(adata, random_state=0, resolution=res, key_added="temp_label")
#         count_unique_leiden = len(list(set(adata.obs["temp_label"])))
#         current_diff = abs(count_unique_leiden - fixed_clus_count)
#         if current_diff < closest_count:
#             closest_count = current_diff
#             closest_res = res
#         if count_unique_leiden == fixed_clus_count:
#             break

#     return closest_res
# def recluster(methods, ncluster):
#     df = pd.read_csv(f"{res_dir}/{methods[0].lower()}_latent.csv", index_col=0)
#     adata = sc.AnnData(X=np.zeros((df.shape[0], 10)))
#     for m in methods:
#         adata.obsm[f"X_{m}"] = np.array(pd.read_csv(f"{res_dir}/{m.lower()}_latent.csv", index_col=0))
#         sc.pp.neighbors(adata, use_rep=f"X_{m}")
#         sc.tl.umap(adata)
#         res = search_resolution(adata, fixed_clus_count=ncluster)
#         sc.tl.leiden(adata, resolution=res, key_added="cluster")
#         print(m, len(set(adata.obs["cluster"])))
#         umap = pd.DataFrame(adata.obsm["X_umap"], columns=["UMAP1", "UMAP2"], index=adata.obs_names)
#         umap.insert(2, "cluster", adata.obs['cluster'].values)
#         umap.to_csv(os.path.join(res_dir, m.lower() + ".csv"))

In [28]:
# recluster(["PRESENT"], ncluster=11)

In [29]:
# metrics = bm.cal_metrics(adata=rna, batch_key="sample", label_key="cell_type_merged",
#                          res_dict=res, methods=methods, verbose=True, rep=1,
#                          min_max_scale=False,
#                          save=f"{result_folder}/metrics.pkl")

In [30]:
with open(f"{result_folder}/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)
metric = metrics[0]

In [31]:
metric = recompute_aggregate_scores(metric)
metric["Total"][:-1] = metric["Batch correction"][:-1] * 0.4 + metric["Bio conservation"][:-1] * 0.6


In [84]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_paired/Multi_batch_lymph"

In [85]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [51]:
# bm.plot_heatmap(metric_df=metric, total_name="Total",
#                 save=f"{figure_save_dir}/summary_heatmap.pdf",
#                 # show_top=7,
#                 # show_bottom=0,
#                 # insert_marker_row = 8,
#                 )

In [79]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [i.obsm["spatial"] for i in split_adata(rna, batch_key="sample")]
spatial = transform_coord(spatial, vertical=False, axis="y", horizontal=False, angle=0)
spatial[1] = transform_coord([spatial[1]], horizontal=True, vertical=True)[0]

In [87]:
spatial_methods = ["COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#5873a4"
bg_dict["ATAC"] = "#5873a4"
bg_dict["Protein"] = "#5873a4"
bg_dict["Batch"] = "#97a4af"
bg_dict["Cell type"] = "#97a4af"
bg_dict["A1"] = "#97a4af"
bg_dict["D1"] = "#97a4af"
bg_dict["Annotation"] = "#97a4af"

In [81]:
from benchmarker import get_scatter_cmap
palette = get_scatter_cmap([str(i) for i in list(range(11))])

In [82]:
palette_annot = {}
for i in range(12):
    palette_annot[sorted(set(adt.obs["cell_type_merged"]))[i]] = get_scatter_cmap([str(i) for i in list(range(12))])[str(i)]

In [ ]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(14, 4.2),
#                 frameon=True,
#                 inner_gs_row=2,
#                 inner_gs_col=1,
#                 size=10,
#                 ncol=7,
#                 xlabel=["Multigrate", "TotalVI", "MOFA2", "PRESENT", "MultiVI",  "CellCharter",  "scMDC"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["Multigrate", "TotalVI", "MOFA2", "PRESENT", "MultiVI",  "CellCharter",  "scMDC"],
#                 outer_row_hspace=0,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 palette=palette,
#                 # inner_col_wspace = -0.12,
#                 ylabel_pad = 0.02,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.013, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods.pdf",
#                 rasterized=True,
#                 )

In [90]:
res["Batch"] = {}
for m in methods:
    res["Batch"][m] = np.array(rna.obs["sample"]).astype(str)

In [92]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=res["Batch"],
#              annot_list=list(rna.obs["cell_type_merged"]),
#              figsize=(14, 4.2),
#              frameon=True,
#              inner_gs_row=2,
#              inner_gs_col=1,
#              size=5,
#              ncol=7,
#              xlabel=["Multigrate", "TotalVI", "MOFA2", "PRESENT", "MultiVI",  "CellCharter",  "scMDC"],
#              only_show_top=False,
#              ylabel=["Batch", "Cell type"],
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["Multigrate", "TotalVI", "MOFA2", "PRESENT", "MultiVI",  "CellCharter",  "scMDC"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#              ylabel_pad=0.0168,
#              xlabel_pad=0.013,
#              outer_row_hspace=0.22,
#              merge=True,
#              merge_margin_size=0.4,
#              palettes=[None, palette_annot],
#              save=f"{figure_save_dir}/umap_methods.pdf"
# )

In [56]:
# bm.plot_legend(["A1", "D1"], marker="o", ncol=1,
# save=f"{figure_save_dir}/batch_legend.pdf",)

In [93]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict={"annot": np.array(adt.obs["cell_type_merged"]).reshape(-1,1)},
#                 figsize=(1.97, 4.2),
#                 frameon=True,
#                 inner_gs_row=2, inner_gs_col=1,
#                 size=10,
#                 ncol=1,
#                 xlabel=["Annotation"],
#                 ylabel=["A1", "D1"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 outer_row_hspace=0.15,
#                 outer_col_wspace=0.1,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.013,
#                 ylabel_pad=0.0165,
#                 save_dpi=600,
#                 # inner_common_camp=True,
#                 palette=get_scatter_cmap([str(i) for i in list(range(12))]),
#                 save=f"{figure_save_dir}/spatial_annot.pdf"
#                 )